# 12.9 · 参数高效微调 / Parameter-Efficient Fine-Tuning (LoRA)

> **课程定位 / Where this fits**
> 第 9 课，**Part 12**。让"在消费级硬件上微调大模型"成为可能的关键技术。
> Lesson 9, **Part 12**. The key tech making "fine-tuning big models on consumer hardware" possible.
>
> 全量微调一个百亿/千亿参数模型，要为**每一个参数**都存梯度和优化器状态(Adam 要存 2 份)——显存需求是模型本身的好几倍，普通人玩不起；而且每个任务都得**存一份完整模型副本**。**参数高效微调(PEFT)** 的思路：**冻结整个预训练模型，只训练极少量新增参数**。最流行的是 **LoRA(低秩适配)**——给权重矩阵加一个**低秩的小增量** $\Delta W = BA$，只训这两个小矩阵。可训练参数能减少**上千倍**，效果却接近全量微调。本课**从零实现 LoRA**并实测。
> Full fine-tuning a billion-parameter model stores gradients and optimizer states for **every parameter** (Adam keeps 2 copies) — several times the model's memory, out of reach for most; plus each task needs a **full model copy**. **PEFT** freezes the entire pretrained model and **trains only a tiny set of new parameters**. The most popular is **LoRA (Low-Rank Adaptation)** — add a **low-rank increment** $\Delta W = BA$ to weight matrices and train only those two small matrices. Trainable params shrink **thousands-fold** with near-full-fine-tune quality. We **implement LoRA from scratch** and measure it.
>
> 💼 **实战/面试视角**："LoRA 原理/为什么低秩有效 / 减少多少参数 / QLoRA / LoRA 加在哪些层" 是 LLM 工程高频。
> 💼 **Practical/interview angle:** "LoRA mechanism / why low-rank works / param reduction / QLoRA / where to apply LoRA" — frequent LLM engineering.

> 📐 **符号约定 / Notation**
> - $W$ —— 预训练权重(冻结) / pretrained weight (frozen)
> - $\Delta W = BA$ —— 低秩增量, $A\in\mathbb{R}^{r\times d}, B\in\mathbb{R}^{d\times r}$, 秩 $r$ 很小 / low-rank update
> - $r$ —— 秩(如 4/8/16) / rank

> 💡 **面试相关 / Interview-relevant**
> - "LoRA 怎么工作(冻结W, 加低秩BA)"（出镜率 ★★★★★）
> - "为什么低秩更新就够了"（★★★★）
> - "LoRA 减少多少可训练参数/显存"（★★★★）
> - "QLoRA 是什么"（★★★★）
> - "LoRA 通常加在哪些权重上"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解为何需要 PEFT(全量微调的显存/存储代价)。
   Understand why PEFT (full fine-tuning's memory/storage cost).
2. 掌握 **LoRA** 的低秩思想与为什么有效。
   Master LoRA's low-rank idea and why it works.
3. **从零实现 LoRALinear** 并注入模型。
   Implement LoRALinear from scratch and inject it.
4. 实测: LoRA 用极少参数达到接近全量微调的效果。
   Measure: LoRA matches full fine-tuning with far fewer params.

## 目录 / TOC
1. [为什么要 PEFT ⭐](#1)
2. [LoRA：低秩适配（从零）⭐](#2)
3. [实测：LoRA vs 全量微调 ⭐](#3)
4. [QLoRA 与变体 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么要 PEFT ⭐ / Why PEFT

全量微调一个大模型，显存里要同时放(面试要点)：
Full fine-tuning a large model must hold in memory (interview point):
- 模型参数本身、所有参数的**梯度**、Adam 优化器的**两个状态(动量+方差)**——加起来约是模型大小的 **3–4 倍**。
  the model, **gradients** for all params, and Adam's **two states (momentum+variance)** — together ~**3–4×** the model size.
- 微调一个 7B 模型(fp16 约 14GB)，全量微调要 **>60GB 显存**——单张消费级卡(如 24GB)根本放不下。
  Fine-tuning a 7B model (~14GB fp16) needs **>60GB** — impossible on a single consumer card (e.g. 24GB).
- 而且每个任务都要**存一整份微调后的模型**(7B×N 个任务)。
  And each task needs a **full fine-tuned copy** (7B × N tasks).

**PEFT 的解法**：**冻结预训练模型(不算它的梯度和优化器状态)，只训练很小一部分新参数**。于是显存主要花在那一小部分上，单卡可训；每个任务只需存那几 MB 的小参数。
**PEFT's solution:** **freeze the pretrained model (no gradients/optimizer states for it), train only a tiny set of new parameters.** Memory goes mostly to that tiny part — single-card feasible; each task only stores a few-MB adapter.


<a id="2"></a>
## 2. LoRA：低秩适配（从零）⭐ / LoRA: Low-Rank Adaptation

**核心观察**：微调时，权重的**改变量 $\Delta W$ 其实"本征秩"很低**——不需要一个满秩的大矩阵来表示这点调整。所以 LoRA 用**两个小矩阵的乘积**来近似 $\Delta W$：
**Key observation:** during fine-tuning, the weight **change $\Delta W$ has low intrinsic rank** — you don't need a full-rank matrix for the adjustment. So LoRA approximates $\Delta W$ as a **product of two small matrices**:

$$W_{\text{new}} = W_{\text{frozen}} + \Delta W, \quad \Delta W = B A, \quad A\in\mathbb{R}^{r\times d_{in}},\ B\in\mathbb{R}^{d_{out}\times r}$$

其中秩 $r$ 很小(如 4、8、16)。原权重 $W$ ($d_{out}\times d_{in}$) **冻结不动**，只训练 $A$ 和 $B$。可训练参数从 $d_{out}\times d_{in}$ 降到 $r\times(d_{in}+d_{out})$——当 $r \ll d$ 时减少**几个数量级**。
where rank $r$ is small (4, 8, 16). The original $W$ ($d_{out}\times d_{in}$) is **frozen**; only $A, B$ train. Trainable params drop from $d_{out}\times d_{in}$ to $r\times(d_{in}+d_{out})$ — orders of magnitude fewer when $r \ll d$.

**两个实现细节(面试)**：① $B$ 初始化为 **0**(所以训练开始时 $\Delta W=0$，模型等于预训练模型，不破坏已有能力)；② 输出乘一个缩放 $\alpha/r$。推理时可把 $BA$ **合并进 $W$**，不增加任何推理延迟。
**Two details (interview):** ① $B$ is initialized to **0** (so $\Delta W=0$ at start — the model equals the pretrained one, not disrupting it); ② scale the output by $\alpha/r$. At inference, $BA$ can be **merged into $W$**, adding zero latency.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math, time, copy
import torch, torch.nn as nn, torch.nn.functional as F
import nltk; nltk.download("gutenberg", quiet=True); from nltk.corpus import gutenberg
sns.set_theme(style="whitegrid"); torch.manual_seed(0)

class LoRALinear(nn.Module):
    """给一个(冻结的)线性层加上低秩适配 ΔW=BA / wraps a frozen Linear with a low-rank update."""
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False    # 冻结原权重 / freeze original weight
        d_in, d_out = base.in_features, base.out_features
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)          # 小矩阵 A / down-projection
        self.B = nn.Parameter(torch.zeros(d_out, r))                # B 初始化为0 → 起始 ΔW=0 / B=0 so ΔW starts at 0
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale  # 原输出 + 低秩增量 / base + low-rank delta

# 参数量对比: 一个 4096×4096 的权重, 全量 vs LoRA(r=8) / param count: full vs LoRA
d = 4096; r = 8
full = d * d
lora = r * (d + d)
print(f"一个 {d}×{d} 权重矩阵:")
print(f"  全量微调可训练参数: {full:,}")
print(f"  LoRA(r={r}) 可训练: {lora:,}  ← 仅为全量的 {100*lora/full:.3f}%(减少 ~{full//lora} 倍)")
print("B 初始化为0 → 训练开始 ΔW=0 → 不破坏预训练能力; 推理可把 BA 合并进 W → 零额外延迟")


<a id="3"></a>
## 3. 实测：LoRA vs 全量微调 ⭐ / Experiment: LoRA vs Full Fine-tuning

复用 12.8 的设定：在奥斯汀文本上预训练一个字符级 GPT，然后到《爱丽丝》上适配——分别用 **LoRA(只训低秩适配)** 和 **全量微调**，对比**可训练参数量**和**效果**。
Reusing 12.8: pretrain a char-level GPT on Austen, then adapt to *Alice* — via **LoRA (train only adapters)** vs **full fine-tuning** — comparing **trainable params** and **quality**.


In [ ]:
pre_text = gutenberg.raw("austen-sense.txt")[:150000].lower(); ft_text = gutenberg.raw("carroll-alice.txt")[:30000].lower()
chars = sorted(set(pre_text+ft_text)); V = len(chars); stoi = {c:i for i,c in enumerate(chars)}
def enc(t): return torch.tensor([stoi[c] for c in t])
CTX = 48
def gb(d, B=64):
    ix = torch.randint(len(d)-CTX-1, (B,)); return torch.stack([d[i:i+CTX] for i in ix]), torch.stack([d[i+1:i+CTX+1] for i in ix])
class MHA(nn.Module):
    def __init__(s,D,H): super().__init__();s.H=H;s.d=D//H;s.qkv=nn.Linear(D,3*D);s.o=nn.Linear(D,D);s.register_buffer("m",torch.tril(torch.ones(CTX,CTX)))
    def forward(s,x):
        B,T,D=x.shape;qkv=s.qkv(x).reshape(B,T,3,s.H,s.d).permute(2,0,3,1,4);q,k,v=qkv
        sc=(q@k.transpose(-2,-1)/math.sqrt(s.d)).masked_fill(s.m[:T,:T]==0,float("-inf"))
        return s.o((F.softmax(sc,-1)@v).transpose(1,2).reshape(B,T,D))
class Blk(nn.Module):
    def __init__(s,D,H): super().__init__();s.a=MHA(D,H);s.n1=nn.LayerNorm(D);s.n2=nn.LayerNorm(D);s.ff=nn.Sequential(nn.Linear(D,4*D),nn.GELU(),nn.Linear(4*D,D))
    def forward(s,x): x=x+s.a(s.n1(x));return x+s.ff(s.n2(x))
class GPT(nn.Module):
    def __init__(s,V,D=96,H=4,L=2): super().__init__();s.tok=nn.Embedding(V,D);s.pos=nn.Embedding(CTX,D);s.blocks=nn.ModuleList([Blk(D,H) for _ in range(L)]);s.ln=nn.LayerNorm(D);s.head=nn.Linear(D,V)
    def forward(s,x):
        T=x.size(1);h=s.tok(x)+s.pos(torch.arange(T))
        for b in s.blocks:h=b(h)
        return s.head(s.ln(h))

pre=enc(pre_text); ft=enc(ft_text); n=int(0.9*len(ft)); ft_tr,ft_va=ft[:n],ft[n:]
torch.manual_seed(0); base=GPT(V); opt=torch.optim.AdamW(base.parameters(),3e-3)
for _ in range(800):                                            # 预训练 / pretrain
    x,y=gb(pre); opt.zero_grad(); F.cross_entropy(base(x).reshape(-1,V),y.reshape(-1)).backward(); opt.step()
pre_state=copy.deepcopy(base.state_dict()); total=sum(p.numel() for p in base.parameters())
def val(m):
    with torch.no_grad(): x,y=gb(ft_va); return F.cross_entropy(m(x).reshape(-1,V),y.reshape(-1)).item()

# LoRA 微调: 注入 LoRA, 冻结其余, 只训 A/B / inject LoRA, freeze rest, train only A/B
torch.manual_seed(1); ml=GPT(V); ml.load_state_dict(pre_state)
for blk in ml.blocks:                                           # 给注意力和FFN的线性层加LoRA / add LoRA to attn+FFN
    blk.a.qkv=LoRALinear(blk.a.qkv); blk.a.o=LoRALinear(blk.a.o); blk.ff[0]=LoRALinear(blk.ff[0]); blk.ff[2]=LoRALinear(blk.ff[2])
for p in ml.parameters(): p.requires_grad=False
for blk in ml.blocks:
    for mod in [blk.a.qkv,blk.a.o,blk.ff[0],blk.ff[2]]: mod.A.requires_grad=True; mod.B.requires_grad=True
trainable=[p for p in ml.parameters() if p.requires_grad]; n_lora=sum(p.numel() for p in trainable)
o=torch.optim.AdamW(trainable,3e-3)
for _ in range(150):
    x,y=gb(ft_tr); o.zero_grad(); F.cross_entropy(ml(x).reshape(-1,V),y.reshape(-1)).backward(); o.step()
lora_val=val(ml)
# 全量微调 / full fine-tuning
torch.manual_seed(1); mf=GPT(V); mf.load_state_dict(pre_state); of=torch.optim.AdamW(mf.parameters(),1e-3)
for _ in range(150):
    x,y=gb(ft_tr); of.zero_grad(); F.cross_entropy(mf(x).reshape(-1,V),y.reshape(-1)).backward(); of.step()
full_val=val(mf)
print(f"总参数 {total:,}")
print(f"LoRA 可训练 {n_lora:,} ({100*n_lora/total:.1f}%), 验证损失 {lora_val:.2f}")
print(f"全量 可训练 {total:,} (100%),        验证损失 {full_val:.2f}")
fig, axes = plt.subplots(1,2,figsize=(11,3.6))
axes[0].bar(["LoRA","全量微调"],[n_lora,total],color=["#39c","#e67"]); axes[0].set_yscale("log"); axes[0].set_title("可训练参数量(对数轴): LoRA 少得多")
axes[1].bar(["LoRA","全量微调"],[lora_val,full_val],color=["#39c","#e67"]); axes[1].set_ylim(min(lora_val,full_val)-0.3,max(lora_val,full_val)+0.3); axes[1].set_title("验证损失: 几乎一样")
plt.tight_layout(); plt.show()
print("LoRA 用极少可训练参数达到接近全量微调的效果 → 省显存/省存储/可单卡微调大模型")


<a id="4"></a>
## 4. QLoRA 与变体 + 小结 ⭐ / QLoRA & Variants

(注：本课小模型上 LoRA 占比约 5%；在**真实的大模型**上，主干巨大而 LoRA 适配器很小，可训练比例常 **<1%** 甚至 0.1%——省得更夸张。)
(Note: on our tiny model LoRA is ~5%; on **real large models**, the backbone is huge and adapters tiny, so the trainable fraction is often **<1%** or even 0.1% — far larger savings.)

**重要变体与实务(面试)**：
**Key variants and practice (interview):**
- **QLoRA**：把**冻结的主干量化到 4-bit**(省显存)，再在其上加 LoRA 适配器(适配器仍用较高精度)。让在**单张 24GB 卡上微调 65B 模型**成为可能——非常实用。
  **QLoRA:** quantize the **frozen backbone to 4-bit** (save memory), then add LoRA adapters on top. Enables fine-tuning a **65B model on a single 24GB card** — highly practical.
- **加在哪些层**：通常加在注意力的 **query 和 value 投影**(经验上性价比最高)，也可加到所有线性层。
  **Where to apply:** typically the attention **query and value projections** (best bang for buck), or all linear layers.
- **多适配器**：一个主干 + 多个不同任务的 LoRA 适配器，**按需切换**(像插拔), 每个只几 MB。
  **Multiple adapters:** one backbone + many task-specific LoRA adapters, **hot-swappable** (each just a few MB).
- **其它 PEFT**：Prefix/Prompt Tuning(训练软提示)、Adapter(插入小模块)、$(IA)^3$ 等。
  **Other PEFT:** Prefix/Prompt Tuning (soft prompts), Adapters (inserted modules), $(IA)^3$, etc.

```
PEFT 动机: 全量微调要存全部参数的梯度+Adam状态(约模型3-4倍显存)+每任务一份副本 → 太贵
LoRA: 冻结W, 加低秩 ΔW=BA(秩r小); 只训A,B; 参数减少几个数量级; 效果≈全量微调
细节: B初始化为0(起始ΔW=0不破坏模型) + 缩放α/r; 推理可把BA合并进W → 零额外延迟
为什么有效: 微调的权重改变量本征秩低, 低秩近似就够
QLoRA: 4-bit量化冻结主干 + LoRA适配器 → 单卡微调65B; 极实用
实务: 常加在注意力q/v投影; 多适配器热插拔; 还有 Prefix/Prompt Tuning 等
```

### 💡 面试速查 / Interview cheat-sheet
1. **LoRA**: 冻结W, 加低秩 BA, 只训 A/B; 参数/显存大降, 效果≈全量。
   LoRA: freeze W, add low-rank BA, train only A/B; big param/memory cut, near-full quality.
2. **为什么低秩够**: 微调的权重改变量本征秩低。
   Why low-rank: fine-tuning weight changes have low intrinsic rank.
3. **细节**: B初始化0(不破坏预训练), 缩放α/r, 推理可合并进W(零延迟)。
   Details: B=0 (no disruption), scale α/r, mergeable at inference (zero latency).
4. **QLoRA**: 4-bit量化主干+LoRA → 单卡微调超大模型。
   QLoRA: 4-bit backbone + LoRA → fine-tune huge models on one card.
5. **加哪里/多适配器**: 常加注意力q/v; 一主干多任务适配器热插拔。
   Where/multi: usually attn q/v; one backbone, swappable per-task adapters.

### 下一节 / Next
**12.10 RLHF / DPO**——预训练+微调让模型"会说话", 但怎么让它说得**符合人类偏好**(有用、诚实、无害)? **RLHF**(基于人类反馈的强化学习)和更简单的 **DPO** 是把基础模型对齐成 ChatGPT 那样"听话助手"的关键。我们会讲清奖励模型、PPO、DPO 的核心思想。
**12.10 RLHF / DPO** — pretrain+fine-tune make a model "talk," but how to make it talk in line with **human preferences** (helpful, honest, harmless)? **RLHF** and the simpler **DPO** are key to aligning a base model into a "helpful assistant" like ChatGPT. We'll cover reward models, PPO, and DPO's core ideas.
